In [1]:
from dotenv import load_dotenv
load_dotenv()
from groq import Groq
import json
from rag_pipeline import retrieve_and_rerank  # 재료 가져오기

client = Groq()
MODEL = "openai/gpt-oss-120b"

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

26
1) len(TEXTS): 26
2) TEXTS 샘플: ['RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.', '환각(hallucination)은 모델이 학습하지 않았거나 근거 없는 내용을 사실인 것처럼 그럴듯하게 지어내는 현상이다.']
3) collection count: 26
4) build_vectorstore:
 def build_vectorstore(texts):
    documents = [Document(page_content=t) for t in texts]

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(documents)

    # 한국어 문서라 multilingual 임베딩 (English 전용 X)
    embeddings = HuggingFaceEmbeddings(
        model_name="paraphrase-multilingual-MiniLM-L12-v2"
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,          # 파라미터명은 embedding (단수)
        collection_name="rag_app",
    )
    return vectorstore

=== 검색 순서 (before) ===
1. RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.
2. 청크를 나눌 때 일부를 겹치게(overlap) 하면 경계에서 잘린 문맥 손실을 줄일 수 있다.
3. LoRA는 원래 가중치는 얼리고 작은 저랭크 행렬만 학습해 파인튜닝 비용을 크게 줄인다.
4. 파인튜닝은 지식 주입보다 말투·형식·도메인 적응에 강하고, 최

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Exception raised in Job[2]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[3]: TimeoutError()


재정렬 OFF: {'faithfulness': 0.1908, 'llm_context_precision_without_reference': 0.4583}
재정렬 ON : {'faithfulness': 0.6351, 'llm_context_precision_without_reference': 0.0000}


In [2]:
# ===== 블록 2: 도구 (실체 함수 + 매핑) =====
def search(query):
    #TODO: retrieve_and_rerank(query) 호출 → 문자열 리스트
    reranked = retrieve_and_rerank(query, first_k=20, top_k=3)
    # TODO: "\n".join 으로 문자열 하나로 합쳐 반환
    result = "\n".join(reranked)
    return result

TOOL_MAP = {"search": search}

In [3]:
print(search("RAG는 환각을 어떻게 줄이는가?"))

RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.
파인튜닝은 사전학습된 모델의 가중치를 특정 데이터로 추가 학습해 행동을 바꾸는 것이다.
환각(hallucination)은 모델이 학습하지 않았거나 근거 없는 내용을 사실인 것처럼 그럴듯하게 지어내는 현상이다.


In [4]:
# ===== 블록 3: 도구 스키마 (calculate 스키마가 템플릿) =====
tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "RAG·파인튜닝·임베딩 등 AI/LLM 지식에 관한 질문에 답할 근거 문서를 검색한다",   # TODO: "언제 검색할지" 판단 문장
            "parameters": {
                # TODO: query(string) 하나 받는 스키마
                    "type": "object",
    "properties": {
        "query": {
            "type": "string",
            "description": "검색할 질문 또는 키워드",   # 이 인자에 뭘 넣어야 하는지
        }
    },
    "required": ["query"],
            },
        },
    }
]

In [9]:
import json

def run_agent(question, max_steps=5):
    messages = [
        {"role": "system", "content": (
    "너는 지식 베이스를 검색해 답하는 어시스턴트다. "
    "규칙:\n"
    "1. 답하기 전에 반드시 search 도구로 근거를 검색한다.\n"
    "2. 검색어는 반드시 한국어로 작성한다 (지식 베이스가 한국어다).\n"
    "3. 질문에 여러 주제가 있으면 각각 따로 검색한다 "
    "(예: 'A와 B 비교' → 'A' 검색, 'B' 검색).\n"
    "4. 검색된 문서 내용에 있는 것만 근거로 답한다. "
    "검색 결과에 없으면 '자료에 없다'고 말한다.\n"
)},
        {"role": "user", "content": question},
    ]
    for step in range(max_steps):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        msg = response.choices[0].message
        print(f"[step {step}] tool_calls: {msg.tool_calls}")   # ★ 이 줄 추가
        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                result = TOOL_MAP[name](**args)
                print(f"   → search('{args.get('query')}') 결과 {len(result)}자")   # ★ 이 줄도
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
        else:
            return msg.content
    return "최대 단계 초과"

In [10]:
print(run_agent("RAG랑 파인튜닝 중에 지식 갱신이 잦을 때 뭐가 나아?"))

[step 0] tool_calls: [ChatCompletionMessageToolCall(id='fc_36b11d78-af6b-4987-a37e-9a0e39175ab8', function=Function(arguments='{"query":"RAG 파인튜닝 지식 갱신 잦을 때 장점"}', name='search'), type='function')]
   → search('RAG 파인튜닝 지식 갱신 잦을 때 장점') 결과 159자
[step 1] tool_calls: None
**RAG가 지식 갱신이 잦은 경우에 더 유리합니다.**  

검색된 자료에 따르면:

- **RAG는 모델 가중치를 바꾸지 않고 외부 지식을 주입**하므로, 최신 문서나 데이터가 추가될 때마다 모델 자체를 다시 학습시킬 필요가 없습니다.  
- 최신 사실을 반영하려면 새로운 문서를 인덱스에 추가하고 검색만 수행하면 되므로 **지식 업데이트 주기가 짧은 도메인에 적합**합니다.  
- 또한, 관련 문서를 검색해 프롬프트에 포함함으로써 **환각(잘못된 생성)을 줄이고 최신 정보를 답변에 직접 반영**할 수 있습니다.

반면에 **파인튜닝**은 모델의 파라미터를 실제로 조정하는 과정이므로, 새로운 지식이 생길 때마다 **재학습(또는 추가 학습) 과정을 거쳐야** 합니다. 이 과정은 시간·자원 소모가 크고, 업데이트 주기가 길어질수록 최신 정보를 반영하기 어렵습니다.  

따라서, 지식이 자주 바뀌는 환경에서는 **RAG**가 더 효율적이고 실용적인 선택이라고 할 수 있습니다. (출처: “RAG는 모델 가중치를 바꾸지 않고 외부 지식을 주입하므로 지식 갱신이 잦은 도메인에 적합한다… 최신 사실 반영에는 RAG가 낫다.”)


In [11]:
run_agent("LoRA랑 QLoRA의 차이가 뭐야?")

[step 0] tool_calls: [ChatCompletionMessageToolCall(id='fc_b271d450-fd7c-4788-8833-116f4d86bd79', function=Function(arguments='{"query":"LoRA QLoRA 차이"}', name='search'), type='function')]
   → search('LoRA QLoRA 차이') 결과 160자
[step 1] tool_calls: None


'LoRA와 QLoRA는 둘 다 **Low‑Rank Adaptation**(저랭크 적응) 기법을 기반으로 하지만, 적용 방식에 차이가 있습니다.\n\n| 구분 | 주요 특징 | 장점 |\n|------|----------|------|\n| **LoRA** | 원래 모델의 가중치를 그대로 두고, 작은 저랭크 행렬(LoRA 매개변수)만 학습합니다. | 파인튜닝에 필요한 연산량·메모리를 크게 줄일 수 있음 |\n| **QLoRA** | 모델을 **4‑bit 양자화**한 뒤에 LoRA를 적용합니다. 즉, 양자화된 모델 위에 저랭크 행렬을 얹어 학습합니다. | 양자화로 인한 메모리 절감에 LoRA를 결합해, 훨씬 적은 VRAM에서도 파인튜닝이 가능 |\n\n요약하면, **LoRA는 가중치를 그대로 유지한 채 저랭크 매개변수만 추가하는 방법**이고, **QLoRA는 모델을 4비트 양자화한 뒤 LoRA를 적용해 메모리 사용량을 더욱 최소화**하는 방법이라는 차이가 있습니다.'